Course instruction but seems out of date

In [0]:
 %sh
 rm -r /dbfs/device_stream
 mkdir /dbfs/device_stream
 wget -O /dbfs/device_stream/devices1.json https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/devices1.json

In [0]:
%sh
if [ -e /dbfs/device_stream ]; then
  echo "device_stream exists"
else
  echo "device_stream does NOT exist"
fi


Suggest alternative

In [0]:
%python
import urllib.request
import json

# Download directly to DBFS without using local filesystem
url = "https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/devices1.json"

# Read the content
with urllib.request.urlopen(url) as response:
    data = response.read().decode('utf-8')

# Clean up old data if it exists
try:
    dbutils.fs.rm("dbfs:/device_stream", recurse=True)
except:
    pass

# Create directory
dbutils.fs.mkdirs("dbfs:/device_stream")

# Write directly to DBFS
dbutils.fs.put("dbfs:/device_stream/devices1.json", data, overwrite=True)

print("File downloaded successfully!")

just want to look at it

In [0]:
%python
# Read as a Spark DataFrame
df = spark.read.json("dbfs:/device_stream/devices1.json")
display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
   
# Create a stream that reads data from the folder, using a JSON schema
inputPath = '/device_stream/'
jsonSchema = StructType([
StructField("device", StringType(), False),
StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)
print("Source stream created...")

In [0]:
# Write the stream to a delta table
delta_stream_table_path = '/delta/iotdevicedata'
checkpointpath = '/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

In [0]:
# Read the data in delta format into a dataframe
df = spark.read.format("delta").load(delta_stream_table_path)
display(df)

In [0]:
df.write.format("delta").saveAsTable("IotDeviceData")

In [0]:
%sql
SELECT * FROM IotDeviceData;

In [0]:
 %sh
 wget -O /dbfs/device_stream/devices2.json https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/devices2.json

In [0]:
%sql
SELECT * FROM IotDeviceData;

In [0]:
deltastream.stop()